# Formix: Kaggle GPU model to local website

Run every cell from top to bottom. Kaggle hosts the passing Qwen 3B Formix adapter. Your local FastAPI backend reaches it through a temporary authenticated Cloudflare tunnel. Production deployment is not changed.

Before starting: enable a Kaggle GPU and Internet. In a new notebook, attach formix-v2-passed-artifacts.zip as an input. Keep this notebook open during the demonstration.

In [ ]:
from pathlib import Path
import os, subprocess

REPO = Path("/kaggle/working/formix")
BRANCH = "codex/formix-model-v2"
if not (REPO / ".git").is_dir():
    subprocess.run([
        "git", "clone", "--branch", BRANCH, "--single-branch",
        "https://github.com/imrishi007/formix.git", str(REPO),
    ], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "origin", BRANCH], check=True)
os.chdir(REPO)
print("Repository ready:", REPO)
subprocess.run(["git", "log", "-1", "--oneline"], check=True)

In [ ]:
import stat, sys, urllib.request

subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=False)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "-r", "ai-training/requirements.txt", "fastapi", "uvicorn", "requests",
], check=True)

CLOUDFLARED = Path("/kaggle/working/cloudflared")
if not CLOUDFLARED.is_file():
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        CLOUDFLARED,
    )
CLOUDFLARED.chmod(CLOUDFLARED.stat().st_mode | stat.S_IEXEC)

import torch
assert torch.cuda.is_available(), "Enable a Kaggle GPU, restart the session, and rerun."
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import json, zipfile

ADAPTER = REPO / "ai-training/output-v2/adapter"
if not (ADAPTER / "adapter_config.json").is_file():
    candidates = list(Path("/kaggle/input").rglob("formix-v2-passed-artifacts.zip"))
    candidates += list(Path("/kaggle/working").glob("formix-v2-passed-artifacts.zip"))
    candidates += list(Path("/kaggle/working").glob("formix-v2-artifacts.zip"))
    if not candidates:
        raise FileNotFoundError("Attach formix-v2-passed-artifacts.zip to this notebook first.")
    archive_path = candidates[0]
    with zipfile.ZipFile(archive_path) as archive:
        for member in archive.infolist():
            target = (REPO / member.filename).resolve()
            if REPO.resolve() not in target.parents and target != REPO.resolve():
                raise RuntimeError("Unsafe archive member: " + member.filename)
        archive.extractall(REPO)
    print("Extracted:", archive_path)

config = json.loads((ADAPTER / "adapter_config.json").read_text())
gate_path = REPO / "ai-training/output-v2/quality_gate.json"
if gate_path.is_file():
    gate = json.loads(gate_path.read_text())
    assert gate.get("passed") is True, "This adapter did not pass the quality gate."
assert (ADAPTER / "adapter_model.safetensors").stat().st_size > 200_000_000
print("Adapter verified:", ADAPTER)
print("Base model:", config["base_model_name_or_path"])

In [ ]:
import re, requests, secrets, threading, time

DEMO_KEY = secrets.token_urlsafe(32)
model_env = os.environ.copy()
model_env["FORMIX_LOCAL_MODEL_KEY"] = DEMO_KEY
MODEL_PROCESS = subprocess.Popen(
    [sys.executable, "ai-training/serve_local.py", "--adapter", str(ADAPTER)],
    cwd=REPO, env=model_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)

ready = False
for line in MODEL_PROCESS.stdout:
    print(line, end="")
    if "Formix model ready at" in line:
        ready = True
        break
    if MODEL_PROCESS.poll() is not None:
        break
if not ready:
    raise RuntimeError("Model service exited with code " + str(MODEL_PROCESS.poll()))

def drain(prefix, stream):
    for output_line in stream:
        print(prefix, output_line, end="")
threading.Thread(target=drain, args=("[model]", MODEL_PROCESS.stdout), daemon=True).start()

for _ in range(30):
    try:
        if requests.get("http://127.0.0.1:8765/health", timeout=5).ok:
            break
    except requests.RequestException:
        time.sleep(2)
else:
    raise RuntimeError("Model service did not become healthy.")

TUNNEL_PROCESS = subprocess.Popen(
    [str(CLOUDFLARED), "tunnel", "--url", "http://127.0.0.1:8765", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
PUBLIC_URL = None
for line in TUNNEL_PROCESS.stdout:
    print(line, end="")
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if match:
        PUBLIC_URL = match.group(0)
        break
    if TUNNEL_PROCESS.poll() is not None:
        break
if not PUBLIC_URL:
    raise RuntimeError("Cloudflare tunnel did not provide a URL.")
threading.Thread(target=drain, args=("[tunnel]", TUNNEL_PROCESS.stdout), daemon=True).start()

for _ in range(30):
    try:
        if requests.get(PUBLIC_URL + "/health", timeout=10).ok:
            break
    except requests.RequestException:
        time.sleep(2)
else:
    raise RuntimeError("Tunnel URL did not become reachable.")

print("\nKAGGLE MODEL READY")
print("Temporary endpoint:", PUBLIC_URL + "/v1")
print("Temporary key:", DEMO_KEY)

In [ ]:
sys.path.insert(0, str(REPO / "ai-training"))
from train import SYSTEM_PROMPT

response = requests.post(
    PUBLIC_URL + "/v1/chat/completions",
    headers={"Authorization": "Bearer " + DEMO_KEY},
    json={
        "model": "formix",
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": "Create a contact form with name, email, and a required message field."},
        ],
        "max_tokens": 700,
    },
    timeout=240,
)
response.raise_for_status()
print(response.json()["choices"][0]["message"]["content"])

print("\nCOPY THESE INTO THE LOCAL BACKEND POWERSHELL WINDOW:")
print("$env:FORMIX_MODEL_BASE_URL = '" + PUBLIC_URL + "/v1'")
print("$env:FORMIX_MODEL_API_KEY = '" + DEMO_KEY + "'")
print("$env:FORMIX_MODEL_NAME = 'formix'")
print("$env:FORMIX_MODEL_MAX_TOKENS = '1200'")

## Start Formix locally

Keep Kaggle running. In a local PowerShell window, paste the four model variables printed above and then run:

    $env:DATABASE_URL = 'sqlite:///./backend/formix-demo.db'
    $env:FORMIX_COMPILER_PATH = '.\forml-compiler\build\forml_compiler.exe'
    python -m uvicorn backend.main:app --host 127.0.0.1 --port 8000

In a second local PowerShell window run:

    $env:NEXT_PUBLIC_API_URL = 'http://127.0.0.1:8000'
    $env:NEXT_PUBLIC_FORMIX_LOCAL_DEMO = '1'
    npm run dev

Open http://localhost:3000 and choose Local Formix model. The local backend still performs native compiler verification and repairs before generated code can be inserted.

In [ ]:
# Run this cell during the presentation. Interrupt it when finished.
try:
    while True:
        model_alive = MODEL_PROCESS.poll() is None
        tunnel_alive = TUNNEL_PROCESS.poll() is None
        health = requests.get(PUBLIC_URL + "/health", timeout=15).status_code
        print(time.strftime("%H:%M:%S"), "model=", model_alive, "tunnel=", tunnel_alive, "health=", health)
        if not model_alive or not tunnel_alive:
            raise RuntimeError("Model or tunnel stopped.")
        time.sleep(60)
except KeyboardInterrupt:
    print("Keep-alive stopped. Run the cleanup cell next.")

In [ ]:
# Cleanup after the presentation.
for process in (globals().get("TUNNEL_PROCESS"), globals().get("MODEL_PROCESS")):
    if process is not None and process.poll() is None:
        process.terminate()
print("Formix Kaggle demo services stopped.")